# FusionMatch — Phase 3: Contrastive Training & Evaluation

This notebook orchestrates the **Two-Phase Contrastive Training Pipeline** for FusionMatch:
- **Loss**: InfoNCE loss with temperature $\tau=0.07$ and hard negative mining.
- **Phase 1 (Warm-Up, Epochs 1–5)**: Frozen SigLIP backbone, training Gated Fusion + Projection Head ($LR = 2 \times 10^{-5}$).
- **Phase 2 (Fine-Tuning, Epochs 6–15)**: Partial fine-tuning of the last 2 transformer blocks ($LR = 2 \times 10^{-6}$) with active hard negative mining.
- **Evaluation**: Validation Pairwise F1-score, Precision@5, and Recall@5.

In [ ]:
import sys
import yaml
from pathlib import Path
import torch
from torch.utils.data import DataLoader

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import FusionMatchModel
from src.data.dataset import FusionMatchDataset, collate_fusion_match_batch
from src.training import FusionMatchTrainer, InfoNCELoss, HardNegativeMiner
from src.training.metrics import compute_pairwise_f1, compute_precision_recall_at_k, evaluate_embeddings
from src.utils.seed import seed_everything

seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load Configurations & DataLoaders

In [ ]:
config_path = project_root / "config" / "base_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

train_manifest = project_root / "data" / "processed" / "manifest_train.csv"
val_manifest = project_root / "data" / "processed" / "manifest_val.csv"

train_dataset = FusionMatchDataset(manifest_path=train_manifest, is_training=True)
val_dataset = FusionMatchDataset(manifest_path=val_manifest, is_training=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=True,
    collate_fn=collate_fusion_match_batch,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
    collate_fn=collate_fusion_match_batch,
    num_workers=0,
)

print(f"Loaded Train Dataset: {len(train_dataset):,} rows ({len(train_loader)} batches)")
print(f"Loaded Val Dataset:   {len(val_dataset):,} rows ({len(val_loader)} batches)")

## 2. Initialize Model & Two-Phase Trainer

In [ ]:
model = FusionMatchModel(
    model_id=config["model"]["backbone_id"],
    embed_dim=config["model"]["embed_dim"],
    use_mock=not torch.cuda.is_available(),  # fast mock on CPU or full SigLIP on GPU
)

trainer = FusionMatchTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config={
        "warmup_epochs": 2 if not torch.cuda.is_available() else config["training"]["warmup_epochs"],
        "total_epochs": 3 if not torch.cuda.is_available() else config["training"]["total_epochs"],
        "lr_head": float(config["training"]["learning_rate"]),
        "lr_backbone": float(config["training"]["learning_rate_backbone"]),
        "temperature": float(config["training"]["temperature"]),
        "checkpoint_dir": str(project_root / "artifacts" / "checkpoints"),
    },
    device=device,
)

summary = model.get_param_budget_summary()
print(f"Total Model Parameters:     {summary['total_params']:,}")
print(f"Trainable Parameters (Warm-Up): {summary['trainable_params']:,}")

## 3. Execute Training Run

In [ ]:
# Execute training
history = trainer.fit()

# Convert history to DataFrame for inspection
df_history = pd.DataFrame(history)
display(df_history)

## 4. Evaluate Validation Split with Best Checkpoint

In [ ]:
from src.utils.io import load_checkpoint

best_ckpt_path = project_root / "artifacts" / "checkpoints" / "best.pt"
if best_ckpt_path.exists():
    ckpt = load_checkpoint(best_ckpt_path, device=device)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded best checkpoint from: {best_ckpt_path}")
    if "metadata" in ckpt:
        print(f"Checkpoint metadata: {ckpt['metadata']}")

final_metrics = trainer.validate()
print("\n=== FINAL VALIDATION METRICS ===")
print(f"Pairwise F1 Score: {final_metrics['val_f1']:.4f}")
print(f"Precision@5:       {final_metrics['p@5']:.4f}")
print(f"Recall@5:          {final_metrics['r@5']:.4f}")